# Gom Cụm Các Bản Nhạc MP3 Trên Bộ Dữ Liệu FMA Small (8.000 bài)

**Sinh viên:** Phạm Tấn Đạt (`079206008013`)

## 1. Kiến trúc giải pháp
1. **Dữ liệu thật:** Bộ dữ liệu **FMA Small** (`8.000` bản nhạc MP3, mỗi bài dài `30 giây`, thuộc `8` thể loại cân bằng trong `fma_metadata/tracks.csv`).
2. **Trích xuất đặc trưng mở rộng (~133 chiều):**
   - **Nhịp điệu:** `Tempo (BPM)`.
   - **Năng lượng & Phổ tần:** `RMS`, `ZCR`, `Spectral Centroid`, `Bandwidth`, `Rolloff`, `Contrast` (`mean`, `std`, `skew`, `kurtosis`).
   - **Hòa âm & Âm sắc:** `Chroma STFT` (12 bậc), `Tonnetz` (6 chiều), `MFCC` (13 hệ số) — mỗi chiều tính đủ 4 thống kê (`mean`, `std`, `skew`, `kurtosis`).
3. **Xử lý dữ liệu lớn:** Trích xuất song song đa nhân (`ProcessPoolExecutor`) kèm **checkpoint CSV** để tiếp tục khi bị ngắt.
4. **Gom cụm & Đánh giá:** `K-Means`, `Agglomerative`, `DBSCAN (auto eps)` và đối chiếu với nhãn thật `genre_top` qua **NMI**, **ARI**, **Purity** và bảng phân bố thể loại theo cụm.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from music_clusterer import (
    scan_audio_files,
    extract_features_batch,
    load_fma_tracks_metadata,
    evaluate_k_range,
    cluster_tracks,
    evaluate_with_ground_truth,
    export_clusters_to_folders,
)

sns.set_theme(style="whitegrid")

## 2. Nạp dữ liệu FMA Small & Metadata (`tracks.csv`)

In [ ]:
fma_dir = Path("fma_small")
tracks_csv = Path("fma_metadata/tracks.csv")
cache_csv = Path("fma_small_features.csv")

if cache_csv.exists():
    df_features = pd.read_csv(cache_csv)
    print(f"Đã nạp từ cache: {len(df_features)} bài hát, {df_features.shape[1]} cột.")
else:
    audio_files = scan_audio_files(str(fma_dir), recursive=True)
    print(f"Tìm thấy {len(audio_files)} tệp MP3 trong {fma_dir}")
    df_features, errors = extract_features_batch(
        audio_files, duration=30.0, n_jobs=-1, checkpoint_path=cache_csv, checkpoint_every=200
    )
    print(f"Trích xuất xong: {len(df_features)} bài | Lỗi: {len(errors)}")

if "genre_top" not in df_features.columns and tracks_csv.exists():
    meta = load_fma_tracks_metadata(tracks_csv)
    df_features = df_features.merge(meta, on="track_id", how="left")

df_features[["track_id", "filename", "genre_top", "duration_sec", "tempo_bpm"]].head(10)

## 3. Tìm số cụm tối ưu (Elbow & Silhouette)

In [ ]:
sample_for_k = df_features.sample(min(1500, len(df_features)), random_state=42)
k_eval = evaluate_k_range(sample_for_k, k_min=2, k_max=12)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(k_eval["k"], k_eval["inertia"], marker="o")
ax1.set_title("Elbow Method (Inertia)")
ax1.set_xlabel("Số cụm (k)")

ax2.plot(k_eval["k"], k_eval["silhouette"], marker="s", color="green")
ax2.set_title("Silhouette Score")
ax2.set_xlabel("Số cụm (k)")
plt.tight_layout()
plt.show()

## 4. Gom cụm & Đánh giá bằng nhãn thật (`genre_top`)

In [ ]:
res = cluster_tracks(df_features, algorithm="kmeans", n_clusters=8, use_pca_for_clustering=True)
print("Chỉ số nội tại:", res.metrics)

gt_scores, crosstab = evaluate_with_ground_truth(res.df, label_col="genre_top")
print("Chỉ số đánh giá với genre_top (NMI / ARI / Purity):", gt_scores)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
plot_df = res.df.sample(min(2000, len(res.df)), random_state=42)
sns.scatterplot(data=plot_df, x="pca_1", y="pca_2", hue="cluster", palette="tab10", s=35, alpha=0.7, ax=ax1)
ax1.set_title("Phân bố cụm trên PCA 2D")

sns.heatmap(crosstab, annot=True, fmt="d", cmap="YlGnBu", ax=ax2)
ax2.set_title("Bảng phân bố Thể loại thật (genre_top) theo Cụm")
plt.tight_layout()
plt.show()

res.cluster_profiles